# Gemma3-4B BP — Checkpoint-125 → GGUF (v2, temiz akış)

**Hedef:** Drive'daki `checkpoint-125` LoRA adapter'ını base modele merge edip Ollama için Q4_K_M GGUF üret.

**Strateji:**
- Unsloth'u kullanma (Gemma3 multimodal save kırık)
- Transformers 4.50.0 + PEFT ile manuel merge
- Base: `unsloth/gemma-3-4b-it` (fp16, public, gated DEĞİL)
- llama.cpp ile HF → GGUF f16 → Q4_K_M

**Önkoşul:** Runtime → Change runtime type → **GPU (T4)**

**Akış:**
1. Bağımlılıkları kur
2. Runtime restart
3. Sürüm doğrula
4. Drive bağla
5. Yolları tanımla
6. Base + adapter merge (PEFT, fp16)
7. llama.cpp kur ve derle
8. HF → GGUF f16
9. Q4_K_M quantize
10. Doğrula
11. (opsiyonel) f16 ara dosyasını sil
12. Ollama Modelfile şablonu

## 1) Bağımlılıkları kur

Unsloth'u tamamen kaldır, Transformers 4.50.0 + PEFT + bitsandbytes yükle.

In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip install -q --force-reinstall --no-deps "transformers==4.50.0"
!pip install -q "tokenizers>=0.21,<0.22" "accelerate>=0.34" "peft>=0.13" "huggingface_hub>=0.26" "bitsandbytes>=0.44" "sentencepiece" "protobuf"
print("\n>>> Kurulum bitti. Sonraki hücreyi çalıştırarak runtime'ı RESTART et.")

## 2) Runtime restart

Aşağıdaki hücre kernel'i öldürür. Restart sonrası **3. adımdan** devam et — 1 ve 2'yi tekrar çalıştırma.

In [ ]:
import os
os.kill(os.getpid(), 9)

## 3) Sürüm doğrula

Restart sonrası buradan başla.

In [ ]:
import transformers, peft, accelerate
print("transformers:", transformers.__version__)  # 4.50.0 olmalı
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)
assert transformers.__version__.startswith("4.50"), "Transformers 4.50 bekleniyordu"
print("\n>>> Sürümler OK.")

## 4) Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5) Yolları tanımla

**Önemli:** `BASE_MODEL` `unsloth/gemma-3-4b-it` (fp16) — `-bnb-4bit` SUFFIX'İ YOK. 4-bit pre-quantized base'i ikinci kez quantize etmeye çalışmak `Blockwise 4bit ... uint8` hatası verir.

In [ ]:
import os

BASE_MODEL = "unsloth/gemma-3-4b-it"   # fp16, public, gated DEĞİL
CKPT       = "/content/drive/MyDrive/bp-finetune/checkpoints/checkpoint-125"
MERGED_DIR = "/content/drive/MyDrive/bp-finetune/merged/gemma3-4b-bp-v1"
GGUF_DIR   = "/content/drive/MyDrive/bp-finetune/gguf"
GGUF_F16   = f"{GGUF_DIR}/gemma3-4b-bp-v1.f16.gguf"
GGUF_Q4    = f"{GGUF_DIR}/gemma3-4b-bp-v1.q4_k_m.gguf"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

assert os.path.isdir(CKPT), f"Checkpoint bulunamadı: {CKPT}"
print("Checkpoint OK :", CKPT)
print("Base model    :", BASE_MODEL)
print("Merged hedef  :", MERGED_DIR)
print("GGUF hedef    :", GGUF_DIR)

## 6) Base + adapter merge (PEFT, fp16)

Base'i fp16'da yükle → LoRA adapter'ı bağla → merge et → diske yaz.

T4'te 4B fp16 ~8GB, VRAM yeter. OOM olursa **6b**'ye geç.

In [ ]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect(); torch.cuda.empty_cache()

print("Base yükleniyor (fp16)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Adapter yükleniyor:", CKPT)
peft_model = PeftModel.from_pretrained(base, CKPT)

print("Merge ediliyor...")
merged = peft_model.merge_and_unload()

print("Diske kaydediliyor:", MERGED_DIR)
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="4GB")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.save_pretrained(MERGED_DIR)

print("\n>>> Merged hazır:", MERGED_DIR)
!ls -lh "$MERGED_DIR"

### 6b) Fallback — CPU-only merge

Yukarıdaki hücre OOM verirse bunu çalıştır. Yavaş (~5-10 dk) ama VRAM'e dokunmaz.

In [ ]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect(); torch.cuda.empty_cache()

print("Base yükleniyor (CPU, fp16)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
)
peft_model = PeftModel.from_pretrained(base, CKPT)
merged = peft_model.merge_and_unload()

merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="4GB")
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(MERGED_DIR)
print("\n>>> CPU merge OK:", MERGED_DIR)
!ls -lh "$MERGED_DIR"

## 7) llama.cpp kur

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

## 8) llama.cpp derle (sadece llama-quantize hedefi)

~3-5 dk sürer. Sadece quantize binary'sini derliyoruz, full build'e gerek yok.

In [ ]:
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF
!cmake --build /content/llama.cpp/build --config Release -j --target llama-quantize

## 9) HF → GGUF (f16)

Merged HF modelini llama.cpp'nin GGUF formatına çevir.

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
  "$MERGED_DIR" \
  --outfile "$GGUF_F16" \
  --outtype f16

## 10) Q4_K_M quantize

Ollama'nın varsayılan `gemma3:4b`'siyle aynı quantization seviyesi.

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
  "$GGUF_F16" \
  "$GGUF_Q4" \
  Q4_K_M

## 11) Doğrula

In [ ]:
import os
for p in [GGUF_F16, GGUF_Q4]:
    if os.path.exists(p):
        gb = round(os.path.getsize(p) / 1e9, 2)
        print(f"OK       {p}  ->  {gb} GB")
    else:
        print(f"MISSING  {p}")

## 12) (Opsiyonel) f16 ara dosyasını sil

Q4_K_M üretildiyse f16 ara dosyasına gerek yok (~8GB yer kazanır).

In [ ]:
import os
if os.path.exists(GGUF_F16):
    os.remove(GGUF_F16)
    print("Silindi:", GGUF_F16)

## 13) Yerelde Ollama'ya yükleme (Colab'da DEĞİL — yerelde yapılacak)

1. Drive'dan `gemma3-4b-bp-v1.q4_k_m.gguf` dosyasını yerel makineye indir.
2. GGUF dosyasının yanına şu içerikle bir `Modelfile` (uzantısız!) oluştur:

```
FROM ./gemma3-4b-bp-v1.q4_k_m.gguf

TEMPLATE """<start_of_turn>user
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
"""

PARAMETER stop "<end_of_turn>"
PARAMETER stop "<start_of_turn>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
```

3. Yerelde terminal:

```bash
ollama create gemma3-bp:v1 -f Modelfile
ollama run gemma3-bp:v1
```

4. Backend `appsettings.json` veya AI servisindeki model adını `gemma3:4b` → `gemma3-bp:v1` olarak değiştir.